# AffectLab — IEMOCAP context and calibration ablation

This notebook tests one predeclared change against the completed utterance-only baseline: the previous three causal dialogue turns with same/other-speaker markers. Each fold fits temperature scaling on its validation session only. Run the four-class experiment first; the six-class context run remains disabled until the benchmark comparison is reviewed.

In [ ]:
!nvidia-smi
import torch

assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import subprocess

from google.colab import auth

PROJECT_ID = 'cat-behaviour-research'
BUCKET = 'affectlab-research-raluca-biras'
PROCESSED_GCS = f'gs://{BUCKET}/data/processed/iemocap-text-v2-context3'
RUNS_GCS = f'gs://{BUCKET}/runs/iemocap-text'
auth.authenticate_user()
subprocess.run(['gcloud', 'config', 'set', 'project', PROJECT_ID], check=True)

In [ ]:
import base64
import json
import os
import sys
from pathlib import Path

from google.colab import userdata

REPO_URL = 'https://github.com/ralucabiras/emotion-aware-role-play-model.git'
REPO_DIR = Path('/content/emotion-aware-role-play-model')
github_token = userdata.get('GITHUB_TOKEN')
if not github_token:
    raise RuntimeError('Add GITHUB_TOKEN to Colab Secrets and enable notebook access.')
basic_auth = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
auth_option = f'http.extraHeader=Authorization: Basic {basic_auth}'
if not REPO_DIR.exists():
    subprocess.run(['git', '-c', auth_option, 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), '-c', auth_option, 'pull', '--ff-only'], check=True)
del github_token, basic_auth, auth_option
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements-ml.txt')], check=True)
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
if hf_token:
    login(token=hf_token, add_to_git_credential=False)

In [ ]:
DATA_ROOT = Path('/content/iemocap-text-v1')
OUTPUT_ROOT = Path('/content/iemocap-context-runs')
subprocess.run(['gcloud', 'storage', 'rsync', '--recursive', PROCESSED_GCS, str(DATA_ROOT)], check=True)
from datasets import load_from_disk

probe = load_from_disk(str(DATA_ROOT / 'benchmark_4' / 'fold-5' / 'dataset'))
assert {'context_text', 'context_turn_count'} <= set(probe['train'].column_names)
assert max(probe['train']['context_turn_count']) <= 3
print('Causal context fields verified; raw dialogue text is not displayed.')

## Context benchmark

Run fold 5 first. If it completes, set `RUN_ALL_CONTEXT_FOLDS = True` and rerun this cell. The primary comparison is the five-fold summary, not one fold.

In [ ]:
SEED = 42
RUN_ALL_CONTEXT_FOLDS = False
folds = range(1, 6) if RUN_ALL_CONTEXT_FOLDS else [5]
context_config = REPO_DIR / 'configs' / 'iemocap_benchmark4_context_deberta_v3_small.json'
context_experiment = 'iemocap_benchmark4_context3_deberta_v3_small'
for fold in folds:
    print(f'\n=== Context benchmark fold {fold} ===')
    subprocess.run(
        [sys.executable, '-m', 'ml.training.train_iemocap_text', '--config', str(context_config),
         '--data-root', str(DATA_ROOT), '--output-root', str(OUTPUT_ROOT), '--fold', str(fold), '--seed', str(SEED)],
        check=True,
    )
    local_fold = OUTPUT_ROOT / context_experiment / f'fold-{fold}'
    subprocess.run(
        ['gcloud', 'storage', 'rsync', '--recursive', str(local_fold), f'{RUNS_GCS}/{context_experiment}/fold-{fold}'],
        check=True,
    )

In [ ]:
if RUN_ALL_CONTEXT_FOLDS:
    context_dir = OUTPUT_ROOT / context_experiment
    subprocess.run([sys.executable, '-m', 'ml.evaluation.summarize_iemocap_folds', str(context_dir)], check=True)
    context_summary = json.loads((context_dir / 'summary.json').read_text())
    subprocess.run(['gcloud', 'storage', 'cp', str(context_dir / 'summary.json'), f'{RUNS_GCS}/{context_experiment}/summary.json'], check=True)
    baseline_file = Path('/content/iemocap-baseline-summary.json')
    subprocess.run(['gcloud', 'storage', 'cp', f'{RUNS_GCS}/iemocap_benchmark4_deberta_v3_small/summary.json', str(baseline_file)], check=True)
    baseline_summary = json.loads(baseline_file.read_text())
    comparison = {
        'baseline_pooled_macro_f1': baseline_summary['pooled']['macro_f1'],
        'context_pooled_macro_f1': context_summary['pooled']['macro_f1'],
        'macro_f1_change': context_summary['pooled']['macro_f1'] - baseline_summary['pooled']['macro_f1'],
        'baseline_mean_raw_ece': baseline_summary['fold_metrics']['test_ece']['mean'],
        'context_mean_raw_ece': context_summary['fold_metrics']['test_ece']['mean'],
        'context_mean_calibrated_ece': context_summary['fold_metrics']['test_calibrated_ece']['mean'],
        'context_pooled_calibrated_ece': context_summary['pooled']['calibrated_ece'],
    }
    display(comparison)
    display(context_summary)
else:
    metrics = json.loads((OUTPUT_ROOT / context_experiment / 'fold-5' / 'metrics.json').read_text())
    display(metrics['validation_metrics'])
    display(metrics['test_metrics'])
    display(metrics['calibration'])

## Six-class context experiment — keep disabled initially

Enable only after reviewing the complete four-class context result. It is justified only if context improves validation/generalization or provides a clearly useful class-level tradeoff.

In [ ]:
RUN_AFFECTLAB_CONTEXT = False
if RUN_AFFECTLAB_CONTEXT:
    affect_config = REPO_DIR / 'configs' / 'iemocap_affectlab6_context_deberta_v3_small.json'
    affect_experiment = 'iemocap_affectlab6_context3_deberta_v3_small_weighted'
    for fold in range(1, 6):
        subprocess.run(
            [sys.executable, '-m', 'ml.training.train_iemocap_text', '--config', str(affect_config),
             '--data-root', str(DATA_ROOT), '--output-root', str(OUTPUT_ROOT), '--fold', str(fold), '--seed', str(SEED)],
            check=True,
        )
        local_fold = OUTPUT_ROOT / affect_experiment / f'fold-{fold}'
        subprocess.run(['gcloud', 'storage', 'rsync', '--recursive', str(local_fold), f'{RUNS_GCS}/{affect_experiment}/fold-{fold}'], check=True)
    affect_dir = OUTPUT_ROOT / affect_experiment
    subprocess.run([sys.executable, '-m', 'ml.evaluation.summarize_iemocap_folds', str(affect_dir)], check=True)
    affect_summary = json.loads((affect_dir / 'summary.json').read_text())
    subprocess.run(['gcloud', 'storage', 'cp', str(affect_dir / 'summary.json'), f'{RUNS_GCS}/{affect_experiment}/summary.json'], check=True)
    display(affect_summary)